Week 4 – ETL in Azure Databricks

In [0]:
from pyspark.sql.functions import col

# 1. Load Cleaned Datasets into Databricks

students_data = [
    (1, "John Doe", "2026-05-01"),
    (2, "Alice Smith", "2026-05-02"),
    (4, "Bob Johnson", "2026-05-04")
]

progress_data = [
    (1, 101, 85),
    (2, 101, 100),
    (4, 103, 10)
]

courses_data = [
    (101, "Python Basics"),
    (102, "Advanced Spark"),
    (103, "Data Modeling")
]

df_students = spark.createDataFrame(students_data, ["student_id", "student_name", "enroll_date"])
df_progress = spark.createDataFrame(progress_data, ["student_id", "course_id", "completion_pct"])
df_courses = spark.createDataFrame(courses_data, ["course_id", "course_name"])


In [0]:
# 2. Join Student and Course Data

final_report_df = df_students.join(df_progress, "student_id", "inner") \
                             .join(df_courses, "course_id", "inner") \
                             .select(
                                 col("student_name"), 
                                 col("course_name"), 
                                 col("enroll_date"), 
                                 col("completion_pct").alias("progress_percentage")
                             )


In [0]:
# 3. Create a final table with: student name, course, enrollment date, progress

print("Final Enriched Student Progress Table:")
final_report_df.show()

Final Enriched Student Progress Table:
+------------+-------------+-----------+-------------------+
|student_name|  course_name|enroll_date|progress_percentage|
+------------+-------------+-----------+-------------------+
| Alice Smith|Python Basics| 2026-05-02|                100|
|    John Doe|Python Basics| 2026-05-01|                 85|
| Bob Johnson|Data Modeling| 2026-05-04|                 10|
+------------+-------------+-----------+-------------------+



In [0]:
#  Save results in Delta or CSV for dashboarding

# Option A: Save as a Delta Table (Recommended for Databricks SQL)
final_report_df.write.format("delta").mode("overwrite").saveAsTable("gold_student_course_summary")
final_report_df.show()

# Option B: Save as a single CSV for external reporting
"""
final_report_df.coalesce(1).write.format("csv") \
               .option("header", "true") \
               .mode("overwrite") \
               .save("/FileStore/tables/final_capstone_report.csv")
"""

+------------+-------------+-----------+-------------------+
|student_name|  course_name|enroll_date|progress_percentage|
+------------+-------------+-----------+-------------------+
| Alice Smith|Python Basics| 2026-05-02|                100|
|    John Doe|Python Basics| 2026-05-01|                 85|
| Bob Johnson|Data Modeling| 2026-05-04|                 10|
+------------+-------------+-----------+-------------------+



'\nfinal_report_df.coalesce(1).write.format("csv")                .option("header", "true")                .mode("overwrite")                .save("/FileStore/tables/final_capstone_report.csv")\n'

WEEK 5 - Pipeline Automation with Azure
`DevOps`

In [0]:
# Run script to generate list of students with less than 50% progress

at_risk_df = spark.table("gold_student_course_summary") \
    .filter(col("progress_percentage") < 50) \
    .select("student_name", "course_name", "progress_percentage")

# Log the count for Azure DevOps logs
count = at_risk_df.count()
print(f"Students with less than 50% progress: {count}")



Students with less than 50% progress: 1


In [0]:
# Output report/log for follow-up
display(at_risk_df)
at_risk_df.coalesce(1).write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weekly_follow_up")

student_name,course_name,progress_percentage
Bob Johnson,Data Modeling,10
